<a href="https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Two findings I would keep in the paper are:

1. The queue is strongest for content that is already visible but underperforming on engagement or CTR. The label comes from the later trend signal, so this is a measured association rather than a causal effect.
2. The learned model improves only modestly over the simple rule baseline on a holdout set. That is still useful for ranking, but the claim should stay directional and decision-support oriented because the validation design is not a causal experiment.


The first finding is about visible-but-weak content being a good refresh candidate. The label comes from the later trend direction, so the paper should say that the observed pattern is associated with later decline and not that the refresh action causes recovery.

The second finding is that a learned model slightly outperforms the hand-built baseline on the holdout set. That claim is supported by the validation design, but it should be framed carefully because the split is client-aware and the task is ranking support, not proving causality.


In [ ]:
import pandas as pd
from pathlib import Path

root = Path.cwd().resolve()
for parent in [root, *root.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'skills').exists():
        root = parent
        break

raw_path = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
feature_path = root / 'data' / 'processed' / 'refresh_feature_vector.csv'

if not feature_path.exists():
    raise FileNotFoundError('Run scripts/01_prepare_features.py first')

frame = pd.read_csv(feature_path)
print('rows', len(frame))
print('target_positive_rate', round(frame['is_declining_label'].mean(), 4))
print('trend_direction_counts')
print(frame['trend_direction'].value_counts().to_dict())


rows 30000
target_positive_rate 0.5421
trend_direction_counts
{'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The honest comparison uses a client-aware holdout split, which is more realistic than a random row-level split because multiple rows can come from the same client and share hidden behavior.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

root = Path.cwd().resolve()
for parent in [root, *root.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'skills').exists():
        root = parent
        break

feature_path = root / 'data' / 'processed' / 'refresh_feature_vector.csv'
frame = pd.read_csv(feature_path)

# Build the same feature set used in the model notebook.
num_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
cat_cols = [
    'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier',
    'word_count_tier', 'impression_tier', 'position_tier'
]

X_num = frame[num_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
X_cat = frame[cat_cols].fillna('unknown').astype(str)
X = pd.concat([
    X_num.reset_index(drop=True),
    pd.get_dummies(X_cat, prefix=cat_cols, dummy_na=False, dtype=float).reset_index(drop=True)
], axis=1)
y = frame['is_declining_label'].astype(int)

# Client-aware holdout split.
client_series = frame['client_id'].fillna('unknown').astype(str)
clients = client_series.drop_duplicates().to_numpy()
random_generator = np.random.default_rng(42)
shuffled_clients = random_generator.permutation(clients)
client_test_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:client_test_count])
client_mask = client_series.isin(test_clients).to_numpy()

train_idx = np.where(~client_mask)[0]
test_idx = np.where(client_mask)[0]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

baseline_scores = frame.iloc[test_idx]['impressions_90d'].to_numpy()
baseline_probs = np.clip((baseline_scores / baseline_scores.max()) if baseline_scores.max() > 0 else np.zeros(len(baseline_scores)), 0, 1)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)
model_probs = model.predict_proba(X_test)[:, 1]

results = pd.DataFrame([
    {'split': 'client_holdout', 'model': 'baseline', 'roc_auc': roc_auc_score(y_test, baseline_probs), 'average_precision': average_precision_score(y_test, baseline_probs)},
    {'split': 'client_holdout', 'model': 'logistic_regression', 'roc_auc': roc_auc_score(y_test, model_probs), 'average_precision': average_precision_score(y_test, model_probs)},
])
print(results)


            split                model   roc_auc  average_precision
0  client_holdout             baseline  0.697239           0.498826
1  client_holdout  logistic_regression  0.700291           0.521542


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The key leakage question is whether a feature is derived from the label or from the same outcome window. If a suspect feature collapses the score, it should be removed from the honest model.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

root = Path.cwd().resolve()
for parent in [root, *root.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'skills').exists():
        root = parent
        break

feature_path = root / 'data' / 'processed' / 'refresh_feature_vector.csv'
frame = pd.read_csv(feature_path)

num_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
cat_cols = [
    'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier',
    'word_count_tier', 'impression_tier', 'position_tier'
]

X_num = frame[num_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
X_cat = frame[cat_cols].fillna('unknown').astype(str)
X = pd.concat([
    X_num.reset_index(drop=True),
    pd.get_dummies(X_cat, prefix=cat_cols, dummy_na=False, dtype=float).reset_index(drop=True)
], axis=1)
y = frame['is_declining_label'].astype(int)

client_series = frame['client_id'].fillna('unknown').astype(str)
clients = client_series.drop_duplicates().to_numpy()
random_generator = np.random.default_rng(42)
shuffled_clients = random_generator.permutation(clients)
client_test_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:client_test_count])
client_mask = client_series.isin(test_clients).to_numpy()

train_idx = np.where(~client_mask)[0]
test_idx = np.where(client_mask)[0]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

leaky_frame = frame.copy()
leaky_frame['leaky_trend_direction'] = leaky_frame['trend_direction'].astype(str)

X_num_leaky = X_num.copy()
X_cat_leaky = pd.concat([X_cat, leaky_frame[['leaky_trend_direction']]], axis=1)
X_leaky = pd.concat([
    X_num_leaky.reset_index(drop=True),
    pd.get_dummies(X_cat_leaky, prefix=['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier', 'leaky_trend_direction'], dummy_na=False, dtype=float).reset_index(drop=True)
], axis=1)

X_train_leaky, X_test_leaky = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]

honest_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
honest_model.fit(X_train, y_train)
leaky_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])
leaky_model.fit(X_train_leaky, y_train)

honest_probs = honest_model.predict_proba(X_test)[:, 1]
leaky_probs = leaky_model.predict_proba(X_test_leaky)[:, 1]

leakage_summary = pd.DataFrame([
    {'model': 'honest', 'roc_auc': roc_auc_score(y_test, honest_probs)},
    {'model': 'with_leaky_feature', 'roc_auc': roc_auc_score(y_test, leaky_probs)},
])
print(leakage_summary)


                model   roc_auc
0              honest  0.700291
1  with_leaky_feature  1.000000


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

A strong but safe rewrite is: “In this dataset, visible pages with weak engagement and CTR were over-represented among later-declining content, and a simple learned model ranked these items slightly better than the rule baseline on a client-aware holdout set.”


In [ ]:
claim_text = (
    'In this dataset, visible pages with weak engagement and CTR were over-represented among later-declining content, '
    'and a simple learned model ranked these items slightly better than the rule baseline on a client-aware holdout set.'
)
print(claim_text)


In this dataset, visible pages with weak engagement and CTR were over-represented among later-declining content, and a simple learned model ranked these items slightly better than the rule baseline on a client-aware holdout set.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.